Day 1, Topic 4: Vectorization – Eliminating Python Loops

## 📘 Topic 4: Vectorization — Eliminating Python Loops

### What Is Vectorization?
**Vectorization** means applying an operation to an **entire array at once** instead of looping over elements one by one. NumPy vectorized operations run in compiled C/Fortran code under the hood — orders of magnitude faster than Python loops.

```python
# ❌ Slow — Python loop
result = []
for i in range(len(a)):
    result.append(a[i] + b[i])

# ✅ Fast — vectorized
result = a + b   # operates on all 1,000,000 elements at once
```

### Element-wise Operations
When you apply an operator between two arrays of the **same shape**, NumPy applies it element-by-element:

```python
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

a + b   → [11, 22, 33, 44]
a * b   → [10, 40, 90, 160]
a ** 2  → [1,  4,  9,  16]
a > 2   → [False, False, True, True]
```

### Universal Functions (ufuncs)
NumPy's **ufuncs** are vectorized math functions that work on entire arrays:
```python
np.sqrt(a)   # square root of every element
np.exp(a)    # e^x for every element
np.log(a)    # natural log
np.abs(a)    # absolute value
```

### Broadcasting — Operating with Scalars
NumPy automatically **broadcasts** (stretches) a scalar to match the array shape:
```python
a + 10    → [11, 12, 13, 14]   # 10 is added to every element
a * 2     → [2,  4,  6,  8]
a > 2.5   → [False, False, True, True]
```

### Aggregate (Reduction) Functions
These collapse an array down to a single value (or along an axis):
```python
np.mean(a)  # average
np.sum(a)   # total
np.std(a)   # standard deviation
np.max(a)   # maximum value
np.min(a)   # minimum value
```

### Axis-wise Operations on 2D Arrays
```python
np.sum(arr, axis=0)  # sum down each column → one value per column
np.sum(arr, axis=1)  # sum across each row  → one value per row
```

### `np.where()` — Vectorized If-Else
```python
np.where(condition, value_if_true, value_if_false)
# Example: apply 10% markup above threshold, 10% discount below
np.where(prices > 20, prices * 1.1, prices * 0.9)
```

### In-place Operations & `out=` Parameter
```python
a += b               # modifies a in-place, avoids creating a new array
np.add(a, b, out=result)  # stores result in pre-allocated array
```
> 💡 **Use `out=`** in tight loops to avoid repeated memory allocation.

In [1]:
import numpy as np

In [2]:
a = np.arange(1_000_000)
b = np.arange(1_000_000)

In [3]:
result = a+b

In [6]:
a = np.array([1, 2, 3, 4])
b = np.array([10, 20, 30, 40])

In [11]:
print(a+b)
print(a*b)
print(a ** 2)

[11 22 33 44]
[ 10  40  90 160]
[ 1  4  9 16]


In [12]:
print(a > 2)

[False False  True  True]


In [13]:
print(np.sqrt(a))

[1.         1.41421356 1.73205081 2.        ]


In [16]:
print(a + 10)
print(a * 2)
print(a > 2.5)

[11 12 13 14]
[2 4 6 8]
[False False  True  True]


In [19]:
np.mean(a)

np.float64(2.5)

In [20]:
np.sum(a)

np.int64(10)

In [21]:
np.std(a)

np.float64(1.118033988749895)

In [22]:
np.max(a)

np.int64(4)

In [23]:
a = np.array([[1, 2], [3, 4], [5, 6]])
b = np.array([[7, 8], [9, 10], [11, 12]])

In [24]:
diff = a - b

In [25]:
sq = diff ** 2

In [28]:
sum_rows = np.sum(sq, axis=1)

In [30]:
distance = np.sqrt(sum_rows)

In [31]:
distance

array([8.48528137, 8.48528137, 8.48528137])

In [33]:
distances = np.sqrt(np.sum((a - b) ** 2, axis=1))

In [34]:
distances

array([8.48528137, 8.48528137, 8.48528137])

In [47]:
a += b

In [48]:
a

array([[15, 18],
       [21, 24],
       [27, 30]])

In [45]:
result = np.empty_like(a)

In [46]:
np.add(a, b, out=result)

array([[15, 18],
       [21, 24],
       [27, 30]])

Activity

In [56]:
def compute_loop(prices, threshold):
    result = np.where(prices > threshold, prices * 1.1, prices * 0.9)
    return result

In [57]:
prices = np.array([10, 20, 30, 40, 50])
threshold = 20

In [58]:
compute_loop(prices, threshold)

array([ 9., 18., 33., 44., 55.])

Day 1, Topic 5: Gauntlet #1 – The Stride Predictor Challenge

## 📘 Topic 5: Gauntlet #1 — The Stride Predictor Challenge

### What This Exercise Tests
This gauntlet challenges you to **predict strides before running code**. Understanding strides is the key to:
- Knowing whether a slice is a view or copy
- Predicting memory access patterns (cache performance)
- Understanding why reshaping/transposing is free (no data moved)

### How to Predict Strides

**Step 1:** Find the element size = `dtype.itemsize` (bytes per element)
- `int32` → 4 bytes
- `float64` → 8 bytes

**Step 2 (C-order):** Work **right to left** through dimensions:
- Last stride = element size
- Each previous stride = next stride × next dimension size

```
Shape (4, 6), dtype=int32:
  col stride = 4 bytes
  row stride = 4 × 6 = 24 bytes
  → strides = (24, 4)
```

**Step 3: Slicing multiplies strides by the step:**
```
arr[::2, ::3].strides = (24×2, 4×3) = (48, 12)
```

**Step 4: Reversing negates strides:**
```
arr[::-1, ::-1].strides = (-24, -4)
```

**Step 5 (F-order):** Work **left to right** — first stride = element size, then multiply forward:
```
Shape (2, 3, 4), dtype=float64, order='F':
  axis-0 stride = 8
  axis-1 stride = 8 × 2 = 16
  axis-2 stride = 16 × 3 = 48
  → strides = (8, 16, 48)
```

### Quick Prediction Cheatsheet

| Operation | Effect on Strides |
|-----------|-------------------|
| Basic slice `[a:b]` | No change (same step=1) |
| Stepped slice `[::n]` | Multiply stride by n |
| Reverse `[::-1]` | Negate stride |
| `.T` (transpose) | Reverse the strides tuple |
| F-order | Reverse the stride-building order |

> 🎯 **Rule of thumb:** If you can express the indexing using only start/stop/step (no lists, no booleans) → it's a **view** with modified strides. Otherwise → **copy**.

In [59]:
arr = np.arange(24, dtype=np.int32).reshape(4, 6)

In [60]:
arr.shape

(4, 6)

In [61]:
arr.strides

(24, 4)

In [62]:
arr = np.arange(24, dtype=np.int32).reshape(4, 6)
sliced = arr[1:3, 2:5]

In [63]:
sliced.shape

(2, 3)

In [64]:
sliced.strides

(24, 4)

In [65]:
arr = np.arange(24, dtype=np.int32).reshape(4, 6)
stepped = arr[::2, ::3]

In [66]:
stepped.shape

(2, 2)

In [67]:
stepped.strides

(48, 12)

In [68]:
arr = np.arange(24, dtype=np.int32).reshape(4, 6)
reversed_arr = arr[::-1, ::-1]

In [69]:
reversed_arr.shape

(4, 6)

In [70]:
reversed_arr.strides

(-24, -4)

In [71]:
arr = np.arange(24, dtype=np.float64).reshape(2, 3, 4, order='F')

In [72]:
arr.shape

(2, 3, 4)

In [73]:
arr.strides

(8, 16, 48)